# 04 — GARCH–Merton playground (Monte Carlo)

Synthetic paths only — no market data.

Discrete GARCH(1,1) returns with Merton jumps:

$$r_t = \mu + \sigma_t Z_t + J_t, \quad \sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

where \(\varepsilon_t = \sigma_t Z_t\), and jumps arrive as compound Poisson (intensity \(\lambda\) per year).

Raise \(\alpha\) for shock persistence / clustering; raise \(\lambda\) for discontinuous jumps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_garch_merton(
    mu, omega, alpha, beta, sigma0,
    lam, mu_j, sigma_j,
    S0, T, n_steps, n_paths, seed=42,
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps  # treat each step as dt years; scale drift/intensity
    # GARCH params assumed on the step frequency used in the slider

    S = np.full(n_paths, S0, dtype=float)
    var = np.full(n_paths, sigma0**2, dtype=float)
    paths = np.empty((n_paths, n_steps + 1))
    vol_paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S
    vol_paths[:, 0] = np.sqrt(var)
    log_rets = np.empty((n_paths, n_steps))
    eps_prev = np.zeros(n_paths)

    for i in range(n_steps):
        var = omega + alpha * eps_prev**2 + beta * var
        var = np.maximum(var, 1e-12)
        sigma = np.sqrt(var)

        z = rng.standard_normal(n_paths)
        eps = sigma * z

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump = np.zeros(n_paths)
        mask = n_jumps > 0
        jump[mask] = (
            n_jumps[mask] * mu_j
            + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
        )

        r = mu * dt + eps + jump
        S = S * np.exp(r)
        paths[:, i + 1] = S
        vol_paths[:, i + 1] = sigma
        log_rets[:, i] = r
        eps_prev = eps

    t = np.linspace(0, T, n_steps + 1)
    return t, paths, vol_paths, log_rets

def plot_garch_merton(
    mu=0.08, omega=1e-6, alpha=0.08, beta=0.90, sigma0=0.012,
    lam=0.4, mu_j=-0.04, sigma_j=0.08,
    S0=100.0, T=1.0, n_steps=252, n_paths=40,
):
    # keep stationarity-ish: alpha + beta < 1
    if alpha + beta >= 0.999:
        beta = max(0.0, 0.998 - alpha)

    t, paths, vol_paths, log_rets = simulate_garch_merton(
        mu, omega, alpha, beta, sigma0, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.8)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean")
    axes[0].set_title("Price paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].plot(t, vol_paths.T, alpha=0.35, lw=0.8)
    axes[1].plot(t, vol_paths.mean(axis=0), color="black", lw=2)
    axes[1].set_title("GARCH conditional σ")
    axes[1].set_xlabel("years")

    axes[2].hist(log_rets.ravel(), bins=80, density=True, alpha=0.75, color="slateblue")
    axes[2].set_title("Step log returns")
    axes[2].set_xlabel("log return")

    fig.suptitle(
        f"ω={omega:.1e}, α={alpha:.2f}, β={beta:.2f}, λ={lam:.2f}  (α+β={alpha+beta:.3f})",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_garch_merton,
    mu=FloatSlider(value=0.08, min=-0.10, max=0.30, step=0.01, description="μ/year"),
    omega=FloatSlider(value=1e-6, min=1e-8, max=5e-5, step=1e-7, description="ω", readout_format=".1e"),
    alpha=FloatSlider(value=0.08, min=0.0, max=0.40, step=0.01, description="α"),
    beta=FloatSlider(value=0.90, min=0.50, max=0.98, step=0.01, description="β"),
    sigma0=FloatSlider(value=0.012, min=0.002, max=0.05, step=0.001, description="σ0"),
    lam=FloatSlider(value=0.40, min=0.0, max=3.0, step=0.1, description="λ"),
    mu_j=FloatSlider(value=-0.04, min=-0.30, max=0.15, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.08, min=0.01, max=0.40, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="T"),
    n_steps=IntSlider(value=252, min=50, max=750, step=10, description="steps"),
    n_paths=IntSlider(value=40, min=5, max=120, step=5, description="paths"),
);

interactive(children=(FloatSlider(value=0.08, description='μ/year', max=0.3, min=-0.1, step=0.01), FloatSlider…